# Import

In [2]:
import requests

# Config

In [1]:
class URL:
    URL = 'https://pokeapi.co/api/v2'
    
    ITEM = f'{URL}/item'
    ITEM_ATTRIBUTE = f'{URL}/item-attribute'
    ITEM_CATEGORIE = f'{URL}/item-category'
    ITEM_FLING_EFFECT = f'{URL}/item-fling-effect'
    ITEM_POCKET = f'{URL}/item-pocket'
    
    POKEMON = f'{URL}/pokemon'
    
    Type = f'{URL}/type'
    

In [ ]:
class utils:
    liste_pokemon_legendaire = [
    'Articuno',
    'Zapdos',
    'Moltres',
    'Mewtwo',
    'Raikou',
    'Entei',
    'Suicune',
    'Lugia',
    'Ho-oh',
    'Regirock',
    'Regice',
    'Registeel',
    'Latias',
    'Latios',
    'Kyogre',
    'Groudon',
    'Rayquaza',
    'Uxie',
    'Mesprit',
    'Azelf',
    'Dialga',
    'Palkia',
    'Giratina',
    'Heatran',
    'Regigigas',
    'Cresselia',
    'Cobalion',
    'Terrakion',
    'Virizion',
    'Tornadus',
    'thundurus',
    'Landorus',
    'Reshiram',
    'Zekrom',
    'kyurem',
    'Xerneas',
    'Yveltal',
    'Zygarde',
    'Type-null',
    'Silvally',
    'Tapu-Koko',
    'tapu-lele',
    'tapu-bulu',
    'tapu-fini',
    'Cosmog',
    'Cosmoem',
    'Solgaleo',
    'Lunala',
    'Necrozma',
    'Zacian',
    'Zamazenta',
    'Eternatus',
    'Kubfu',
    'Urshifu',
    'Regidrago',
    'Calyrex',
    'Glastrier',
    'Spectrier',
    'Enamorus',
    'Wo-chien',
    'Chien-pao',
    'Ting-lu',
    'chi-yu',
    'Koraidon',
    'Miraidon',
    'Okidogi',
    'Munkidori',
    'Fezandipiti',
    'Ogerpon',
    'Terapagos'
]
    liste_pokemon_mythiques = [
        # Gen 1
        "Mew", 
        # Gen 2
        "Celebi", 
        # Gen 3
        "Jirachi", "Deoxys", 
        # Gen 4
        "Phione", "Manaphy", "Darkrai", "Shaymin", "Arceus", 
        # Gen 5
        "Victini", "Keldeo", "Meloetta", "Genesect", 
        # Gen 6
        "Diancie", "Hoopa", "Volcanion", 
        # Gen 7
        "Magearna", "Marshadow", "Zeraora", "Meltan", "Melmetal", 
        # Gen 8
        "Zarude", 
        # Gen 9
        "Pecharunt"
    ]

# Utils

In [13]:
def get_full_collection(url):
    full_results = []

    while url:
        response = requests.get(url, timeout=5)
        response.raise_for_status()

        data = response.json()
        full_results.extend(data["results"])

        url = data["next"]

    return {
        "count": len(full_results),
        "results": full_results
    }

In [6]:
def write_names_to_file(url, folder, filename):
    
    full_collection_item = get_full_collection(url)
    
    names = [item["name"] for item in full_collection_item["results"]]
    
    path = folder

    file_path = f'{path}/{filename}'

    with open(file_path, "w", encoding="utf-8") as f:
        for name in names:
            f.write(f"{name}\n")

# Item

In [4]:
def is_holdable(item):
    for attribute in item['attributes']:
        if attribute['name']=='holdable':
            return True
    return False

In [ ]:
def write_item_infos_to_ttl_file(url, folder, filename):

    full_collection_item = get_full_collection(url)

    names = [item["name"] for item in full_collection_item["results"]]

    file_path = f"{folder}/{filename}"

    with open(file_path, "w", encoding="utf-8") as f:
        for name in names:

            istype, types, egg_groups, evolution_infos = get_poke_infos(name)
            
            lines = [
                f"### http://www.semanticweb.org/arthu/ontologies/2026/0/OntoPokemon#{name}",
                f":{name} rdf:type owl:NamedIndividual ,",
                f"                             :{istype} ;",
                f"                    :BelongsToEggGroup {', '.join(f':{eg}' for eg in egg_groups)} ;"
            ]

            if evolution_infos and evolution_infos[1]:
                lines.append(f"                    :EvolvesTo {', '.join(f':{evo}' for evo in evolution_infos[1])} ;")

            if evolution_infos and evolution_infos[0]:
                lines.append(f"                    :IsEvolutionOf :{evolution_infos[0]} ;")

            lines.extend([
                f"                    :Pokemon_IsTypeOf {', '.join(f':{t}' for t in types)} ;",
                f'                    :nom "{name}" .\n\n',
                ""
            ])

            f.write("\n".join(lines))

In [ ]:
write_item_infos_to_ttl_file(URL.ITEM, 'thing_ttl', 'item.txt')

# Pokemon

In [7]:
def est_legendaire_ou_fabuleux(nom_pokemon):
    
    legendaires = utils.liste_pokemon_legendaire
    fabuleux = utils.liste_pokemon_mythiques
    
    nom = nom_pokemon.lower()

    for p in legendaires:
        if p.lower() in nom:
            return "Legendaire"

    for p in fabuleux:
        if p.lower() in nom:
            return "Fabuleux"

    return "Pokemon"

In [8]:
def get_evolution_infos(poke, evolution_chain, is_evolution_of=None):

    nom = evolution_chain["species"]["name"]

    # Pokémon trouvé
    if nom == poke:
        evolutions = [evo['species']['name'] for evo in evolution_chain.get("evolves_to", [])]
        return is_evolution_of, evolutions

    # Descente récursive CORRECTE
    for evo in evolution_chain.get("evolves_to", []):
        infos = get_evolution_infos(poke, evo, nom)
        if infos:
            return infos

    return None

In [27]:
def get_poke_infos(name):
    name_url = f'{URL.POKEMON}/{name}/'
    
    istype = est_legendaire_ou_fabuleux(name)

    reponse = requests.get(name_url, timeout=5)
    reponse.raise_for_status()

    poke = reponse.json()
    
    species_url = poke['species']['url']
    
    reponse = requests.get(species_url, timeout=5)
    reponse.raise_for_status()
    
    species = reponse.json()
    
    reponse = requests.get(species['evolution_chain']['url'], timeout=5)
    reponse.raise_for_status()

    evolution_chain = reponse.json()
    
    egg_groups = [eg['name'] for eg in species['egg_groups']]
    
    types = [type['type']['name'] for type in poke['types']]
    
    evolution_infos = get_evolution_infos(name, evolution_chain['chain'])
  
    return istype, types, egg_groups, evolution_infos


In [ ]:
def write_poke_infos_to_ttl_file(url, folder, filename):

    full_collection_item = get_full_collection(url)

    names = [item["name"] for item in full_collection_item["results"]]

    file_path = f"{folder}/{filename}"

    with open(file_path, "w", encoding="utf-8") as f:
        for name in names:

            istype, types, egg_groups, evolution_infos = get_poke_infos(name)
            
            lines = [
                f"### http://www.semanticweb.org/arthu/ontologies/2026/0/OntoPokemon#{name}",
                f":{name} rdf:type owl:NamedIndividual ,",
                f"                             :{istype} ;",
                f"                    :BelongsToEggGroup {', '.join(f':{eg}' for eg in egg_groups)} ;"
            ]

            if evolution_infos and evolution_infos[1]:
                lines.append(f"                    :EvolvesTo {', '.join(f':{evo}' for evo in evolution_infos[1])} ;")

            if evolution_infos and evolution_infos[0]:
                lines.append(f"                    :IsEvolutionOf :{evolution_infos[0]} ;")

            lines.extend([
                f"                    :Pokemon_IsTypeOf {', '.join(f':{t}' for t in types)} ;",
                f'                    :nom "{name}" .\n\n',
                ""
            ])

            f.write("\n".join(lines))

In [11]:
# ###  http://www.semanticweb.org/arthu/ontologies/2026/0/OntoPokemon#Pokemon_Bulbizarre
# :Pokemon_Bulbizarre rdf:type owl:NamedIndividual ,
#                              :Pokemon ;
#                     :BelongsToEggGroup :GroupeOeuf_Monstrueux ,
#                                        :GroupeOeuf_Vegetal ;
#                     :EvolvesTo :Pokemon_Herbizarre ;
#                     :EvolvesWithTrigger :TriggerEvolution_Bulbizarre_To_Herbizarre ;
#                     :Pokemon_IsTypeOf :Type_Plante ,
#                                       :Type_Poison ;
#                     :nom "Bulbizarre" .

In [12]:
# write_names_to_file(URL.POKEMON, 'thing', 'pokemon.txt')

In [ ]:
write_poke_infos_to_ttl_file(URL.POKEMON, 'thing_ttl', 'pokemon.txt')

# 4 min 17 s presque toute la liste des poke (-~150)

# Types

In [15]:
def get_type_infos(name):
    name_url = f'{URL.Type}/{name}/'

    reponse = requests.get(name_url, timeout=5)
    reponse.raise_for_status()

    type = reponse.json()
    
    double_damage_from = [t['name'] for t in type['damage_relations']['double_damage_from']]
    
    double_damage_to = [t['name'] for t in type['damage_relations']['double_damage_to']]
    
    half_damage_from = [t['name'] for t in type['damage_relations']['half_damage_from']]
    
    half_damage_to = [t['name'] for t in type['damage_relations']['half_damage_to']]
    
    no_damage_from = [t['name'] for t in type['damage_relations']['no_damage_from']]
    
    no_damage_to = [t['name'] for t in type['damage_relations']['no_damage_to']]
  
    return double_damage_from, double_damage_to, half_damage_from, half_damage_to, no_damage_from, no_damage_to


In [26]:
def write_types_infos_to_ttl_file(url, folder, filename):

    full_collection_item = get_full_collection(url)

    names = [item["name"] for item in full_collection_item["results"]]

    file_path = f"{folder}/{filename}"

    with open(file_path, "w", encoding="utf-8") as f:
        for name in names:

            istype = 'Type_pokemon'

            ddf, ddt, hdf, hdt, ndf, ndt = get_type_infos(name)
            
            lines = [
                f"### http://www.semanticweb.org/arthu/ontologies/2026/0/OntoPokemon#{name}",
                f":{name} rdf:type owl:NamedIndividual ,",
            ]
            
            if not any((ddf, hdf, ndf)):
                lines.append(f"                             :{istype} .\n\n")
            else:
                lines.append(f"                             :{istype} ;")

            if ddf:
                if hdf or ndf:
                    lines.append(f"                    :WeakTo {', '.join(f':{t}' for t in ddf)} ;")
                else :
                    lines.append(f"                    :ResistantTo {', '.join(f':{t}' for t in hdf)} .\n\n")

            if hdf:
                if ndf:
                    lines.append(f"                    :ResistantTo {', '.join(f':{t}' for t in hdf)} ;")
                else:
                    lines.append(f"                    :ResistantTo {', '.join(f':{t}' for t in hdf)} .\n\n")
                
            if ndf:
                lines.append(f"                    :ImmuneTo {', '.join(f':{t}' for t in ndf)} .\n\n")
                
            
            f.write("\n".join(lines))

In [ ]:
# ###  http://www.semanticweb.org/arthu/ontologies/2026/0/OntoPokemon#Type_Poison
# :Type_Poison rdf:type owl:NamedIndividual ,
#                       :Type_pokemon ;
#              :ResistantTo :Type_Combat ,
#                           :Type_Fee ,
#                           :Type_Insecte ,
#                           :Type_Plante ,
#                           :Type_Poison ;
#              :WeakTo :Type_Psy ,
#                      :Type_Sol .

In [27]:
write_types_infos_to_ttl_file(URL.Type, 'thing_ttl', 'type.txt')

# tests

In [ ]:
# leg, fab = write_names_to_ttl_file(URL.POKEMON, 'thing_ttl', 'pokemon.txt')

In [ ]:
# for pokemon in liste_pokemon_legendaire:
#     find = False
#     for legendaires in leg:
#         if pokemon.lower() in legendaires:
#             find = True
#     if find == False:
#         print(pokemon)
# for pokemon in liste_pokemon_mythiques:
#     find = False
#     for mytiques in fab:
#         if pokemon.lower() in mytiques:
#             find = True
#     if find == False:
#         print(pokemon)